In [1]:
from unc_handling import UG_prompter
from DataLoader import DataLoader
from segmentation import Segmentation
from evaluation import Evaluator

root = r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development"


data = DataLoader(parentfolder=root,subject_nr=18,volume_of_interest="CTVT",verbose=True)
unc_handler = UG_prompter(data=data)
seg_handler = Segmentation(data=data)

Loaded subject newAcq_ceef36f726ebdaed with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image spacing (z, y, x): (2.5, 0.46880000829696655, 0.46880000829696655)
Initilialized UG_prompter for subject 18 with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024).
Initialized Segmentation for subject 18 with volume of interest 'CTVT'
Mask shape: (88, 1024, 1024)
Building SAM predictor from checkpoint: checkpoints/MedSAM2_latest.pt


c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\modeling\sam\transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()


In [2]:
unc_handler.threshold_uncertainty_map(unc_threshold=0.019496,target_mm=3.0,step_fraction=0.025)

array([[[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, Fal

In [3]:
methods_available = ["raycast", "local_normals"]
method = methods_available[0]

unc_handler.compute_band_thickness(method=method)

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, np.float64(6.778066786626975), np.float64(9.747133505841097), np.float64(5.264233426501353), np.float64(3.5127445066140757), np.float64(2.431900043040514), np.float64(2.086811148044136), np.float64(1.8003222540848784), np.float64(1.6342889178130362), np.float64(1.6342889178130362), np.float64(1.6310333621998627), np.float64(1.7221889193687174), np.float64(1.8296222546034389), np.float64(1.9142667005459468), np.float64(2.0054222577148013), np.float64(2.0412333694597087), np.float64(2.164944482760297), np.float64(2.584911156859663), np.float64(4.157344518022405), np.float64(6.377633446206649), np.float64(4.544755635990037), np.float64(2.405855598135127), np.float64(2.7639667155841985), np.float64(4.5154556354714765), np.float64(9.200200162827969), 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0

In [4]:
prompt_method = methods_available[0]

prompts = unc_handler.generate_prompts_nietjes(unc_band_thr_mm=3.0,
        interpix_dist=3,
        pixel_interval=1,
        angle_step=10,
        method=prompt_method)

print(prompts)

{29: {'points': array([[503.     , 515.     ],
       [529.0973 , 515.     ],
       [503.     , 515.     ],
       [528.947  , 519.57513],
       [503.     , 515.     ],
       [527.75836, 524.0113 ],
       [503.     , 515.     ],
       [525.8174 , 528.17365],
       [503.     , 515.     ],
       [523.18317, 531.93567],
       [503.     , 515.     ],
       [519.93567, 535.18317],
       [503.     , 515.     ],
       [516.17365, 537.8174 ],
       [503.     , 515.     ],
       [512.0113 , 539.75836],
       [503.     , 515.     ],
       [507.57516, 540.947  ],
       [503.     , 515.     ],
       [503.     , 541.0973 ],
       [503.     , 515.     ],
       [498.42484, 540.947  ],
       [503.     , 515.     ],
       [493.9887 , 539.75836],
       [503.     , 515.     ],
       [489.82635, 537.8174 ],
       [503.     , 515.     ],
       [486.0643 , 535.18317],
       [503.     , 515.     ],
       [482.81683, 531.93567],
       [503.     , 515.     ],
       [480.1826 , 528.

In [5]:
# PLOTTING FUNCTION FOR IMAGE, SEGMENTATION, UNCERTAINTY, PROMPTS, NORMALS/RAYS

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from ipywidgets import interact, IntSlider, Checkbox
from matplotlib.colors import ListedColormap


def plot_ug_prompts_interactive(
    ug_instance,
    alpha_seg=0.35,
    alpha_unc=0.35,
    figsize=(7, 7),
    show_only_prompted_slices=False,
    unc_mode="binary",
    zoom_fraction=0.25,
):
    """
    Interactive slice viewer with optional zoom.

    Toggleable overlays:
    - uncertainty
    - vectors (normals OR rays)
    """

    if not hasattr(ug_instance, "img"):
        raise AttributeError("The instance has no attribute 'img'.")
    if not hasattr(ug_instance, "mask"):
        raise AttributeError("The instance has no attribute 'mask'.")
    if not hasattr(ug_instance, "prompts_by_slice"):
        raise AttributeError(
            "The instance has no attribute 'prompts_by_slice'. "
            "Run a prompt generation function first."
        )

    img = ug_instance.img
    mask = ug_instance.mask
    prompts_by_slice = ug_instance.prompts_by_slice

    if hasattr(ug_instance, "unc_map_bin"):
        unc = ug_instance.unc_map_bin.astype(bool)
    elif hasattr(ug_instance, "unc_map"):
        unc = ug_instance.unc_map
    else:
        unc = None

    n_slices = img.shape[0]

    if show_only_prompted_slices:
        slice_list = sorted(prompts_by_slice.keys())
        if len(slice_list) == 0:
            print("No prompted slices found.")
            return
    else:
        slice_list = list(range(n_slices))

    def _plot(slice_idx_in_list, show_uncertainty, show_vectors, zoom):
        z = slice_list[slice_idx_in_list]

        H, W = img[z].shape

        # --- define crop ---
        if zoom:
            h_half = int(H * zoom_fraction / 2)
            w_half = int(W * zoom_fraction / 2)
            cy, cx = H // 2, W // 2
            y0, y1 = cy - h_half, cy + h_half
            x0, x1 = cx - w_half, cx + w_half
        else:
            y0, y1 = 0, H
            x0, x1 = 0, W

        def shift_points(points):
            return np.column_stack([
                points[:, 0] - x0,
                points[:, 1] - y0
            ])

        fig, ax = plt.subplots(figsize=figsize)

        # --- image ---
        img_crop = img[z][y0:y1, x0:x1]
        ax.imshow(img_crop, cmap="gray")

        # --- segmentation ---
        seg_crop = mask[z][y0:y1, x0:x1]
        seg_overlay = np.ma.masked_where(~seg_crop, seg_crop)
        ax.imshow(seg_overlay,cmap=ListedColormap(["red"]), alpha=alpha_seg)

        # --- uncertainty ---
        if show_uncertainty and unc is not None:
            unc_slice = unc[z][y0:y1, x0:x1]

            if unc_mode == "binary":
                unc_overlay = np.ma.masked_where(~unc_slice, unc_slice)
            else:
                unc_overlay = np.ma.masked_where(unc_slice <= 0, unc_slice)

            ax.imshow(unc_overlay, cmap = ListedColormap(["deepskyblue"]), alpha=alpha_unc)

        # --- prompts ---
        prompt_data = prompts_by_slice.get(z, {})

        points = prompt_data.get("points", None)
        point_labels = prompt_data.get("point_labels", None)

        if points is not None and point_labels is not None:
            points = np.asarray(points)
            point_labels = np.asarray(point_labels)

            if len(points) > 0:
                pos = points[point_labels == 1]
                neg = points[point_labels == 0]

                if len(pos) > 0:
                    pos = shift_points(pos)
                    ax.scatter(
                        pos[:, 0], pos[:, 1],
                        s=5, c="lightgreen", marker="+", label="positive"
                    )

                if len(neg) > 0:
                    neg = shift_points(neg)
                    ax.scatter(
                        neg[:, 0], neg[:, 1],
                        s=5, c="lightcoral", marker="x", label="negative"
                    )

        # --- boxes ---
        boxes = prompt_data.get("boxes", None)
        if boxes is None:
            boxes = prompt_data.get("bbox", None)

        if boxes is not None:
            boxes = np.asarray(boxes)
            if boxes.ndim == 1:
                boxes = boxes[None, :]

            for i, (bx0, by0, bx1, by1) in enumerate(boxes):
                rect = Rectangle(
                    (bx0 - x0, by0 - y0),
                    bx1 - bx0,
                    by1 - by0,
                    fill=False,
                    edgecolor="cyan",
                    linewidth=1,
                    label="box" if i == 0 else None,
                )
                ax.add_patch(rect)

        # --- VECTORS: normals OR rays ---
        if show_vectors:

            # ---------- NORMALS ----------
            normals_data = getattr(ug_instance, "normals_by_slice", {}).get(z, None)

            if normals_data is not None:
                mids = np.asarray(normals_data["midpoints"])  # (y, x)

                if "outer_normals" in normals_data:
                    norms = np.asarray(normals_data["outer_normals"])
                elif "normals" in normals_data:
                    norms = np.asarray(normals_data["normals"])
                else:
                    norms = None

                if norms is not None:
                    mask_crop = (
                        (mids[:, 0] >= y0) & (mids[:, 0] < y1) &
                        (mids[:, 1] >= x0) & (mids[:, 1] < x1)
                    )

                    mids = mids[mask_crop]
                    norms = norms[mask_crop]

                    if len(mids) > 0:
                        mids_plot = np.column_stack([
                            mids[:, 1] - x0,
                            mids[:, 0] - y0,
                        ])

                        arrow_length = 15

                        norms_plot = norms.copy().astype(float)
                        norms_plot /= (
                            np.linalg.norm(norms_plot, axis=1, keepdims=True) + 1e-8
                        )
                        norms_plot *= arrow_length

                        ax.quiver(
                            mids_plot[:, 0],
                            mids_plot[:, 1],
                            norms_plot[:, 1],
                            norms_plot[:, 0],
                            color="cyan",
                            scale=1,
                            width=0.003,
                            alpha=0.7,
                            angles="xy",
                            scale_units="xy",
                            label="normals",
                        )

            # ---------- RAYS ----------
            rays_data = getattr(ug_instance, "rays_by_slice", {}).get(z, None)

            if rays_data is not None:
                origin_yx = np.asarray(rays_data["origin_yx"], dtype=float)
                dirs_yx = np.asarray(rays_data["directions_yx"], dtype=float)

                seg_mm = np.asarray(
                    rays_data.get("seg_mm", np.ones(len(dirs_yx)) * 20),
                    dtype=float
                )

                spacing_y = ug_instance.img_spacing[1]
                spacing_x = ug_instance.img_spacing[2]

                # --- normalize directions ---
                dirs_norm = dirs_yx / (np.linalg.norm(dirs_yx, axis=1, keepdims=True) + 1e-8)

                # --- convert origin to plot coords ---
                origin_plot = np.array([
                    origin_yx[1] - x0,
                    origin_yx[0] - y0,
                ])

                ax.scatter(
                    origin_plot[0],
                    origin_plot[1],
                    s=20,
                    c="cyan",
                    marker="o",
                    label="ray origin",
                )

                # --- control how far arrows extend ---
                extend_mm = 20.0  # extra length beyond segmentation

                # --- convert lengths to pixel space ---
                lengths_px = np.column_stack([
                    (seg_mm + extend_mm) / spacing_y,  # y
                    (seg_mm + extend_mm) / spacing_x,  # x
                ])

                # --- compute arrow vectors in pixel space ---
                vecs_y = dirs_norm[:, 0] * lengths_px[:, 0]
                vecs_x = dirs_norm[:, 1] * lengths_px[:, 1]

                # --- plot arrows ---
                ax.quiver(
                    np.full_like(vecs_x, origin_plot[0]),  # x start
                    np.full_like(vecs_y, origin_plot[1]),  # y start
                    vecs_x,                                # dx
                    vecs_y,                                # dy
                    color="cyan",
                    angles="xy",
                    scale_units="xy",
                    scale=1,
                    width=0.003,
                    alpha=0.6,
                    label="rays"
                )

        # --- title ---
        has_prompt = z in prompts_by_slice
        title = f"Slice {z} | prompted: {has_prompt}"

        if (
            hasattr(ug_instance, "band_thickness_per_slice")
            and z < len(ug_instance.band_thickness_per_slice)
        ):
            band = ug_instance.band_thickness_per_slice[z]
            title += f" | band: {band:.2f} mm"

        ax.set_title(title)
        ax.set_axis_off()

        handles, labels = ax.get_legend_handles_labels()
        if handles:
            unique = dict(zip(labels, handles))
            ax.legend(unique.values(), unique.keys(), loc="upper right")

        plt.tight_layout()
        plt.show()

    interact(
        _plot,
        slice_idx_in_list=IntSlider(
            min=0,
            max=len(slice_list) - 1,
            step=1,
            value=0,
            description="slice",
        ),
        show_uncertainty=Checkbox(value=False, description="uncertainty"),
        show_vectors=Checkbox(value=False, description="vectors"),
        zoom=Checkbox(value=False, description="zoom"),
    )

In [6]:
plot_ug_prompts_interactive(unc_handler,unc_mode="abs")

interactive(children=(IntSlider(value=0, description='slice', max=87), Checkbox(value=False, description='unce…

In [7]:
seg_handler.load_dense_prompt()
seg_handler.add_prompt_dict(prompt_dict=prompts)
seg_handler.check_loaded_prompts()
seg_handler.construct_prompts()
seg_handler.run_segmentation()

Added new prompt dict with slices: [29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52]
In total, 2 prompt dict(s) loaded.
0th prompt dict contains prompts for slices: [29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52]
1th prompt dict contains prompts for slices: [29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52]
To combine prompts for segmentation, run construct_prompts
Device: cuda
Volume shape (D,H,W): (88, 1024, 1024)
Using prompts from slices: [29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52]
Adding prompt(s) on slice 29


c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\sam2_video_predictor_npz.py:965: UserWarning: cannot import name '_C' from 'sam2' (c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


Adding prompt(s) on slice 30
Adding prompt(s) on slice 31
Adding prompt(s) on slice 32
Adding prompt(s) on slice 33
Adding prompt(s) on slice 34
Adding prompt(s) on slice 35
Adding prompt(s) on slice 36
Adding prompt(s) on slice 37
Adding prompt(s) on slice 38
Adding prompt(s) on slice 39
Adding prompt(s) on slice 40
Adding prompt(s) on slice 41
Adding prompt(s) on slice 42
Adding prompt(s) on slice 43
Adding prompt(s) on slice 44
Adding prompt(s) on slice 45
Adding prompt(s) on slice 46
Adding prompt(s) on slice 47
Adding prompt(s) on slice 48
Adding prompt(s) on slice 49
Adding prompt(s) on slice 50
Adding prompt(s) on slice 51
Adding prompt(s) on slice 52
Forward propagation (default)...


propagate in video: 100%|██████████| 59/59 [00:12<00:00,  4.85it/s]


Backward propagation (default)...


propagate in video: 100%|██████████| 30/30 [00:09<00:00,  3.03it/s]


array([[[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       ...,

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 

In [8]:
# PLOTTING FUNCTION FOR GT VS DENSE MASK VS UPDATED SEGMENTATION + PROMPTS

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Checkbox
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle


def plot_segmentation_overlays_interactive(
    seg_instance,
    figsize=(7, 7),
    alpha_gt=0.45,
    alpha_dense_mask=0.35,
    alpha_seg=0.45,
    zoom_fraction=0.25,
):
    """
    Interactive viewer for image, ground truth, dense prompt mask,
    newly generated segmentation, point prompts, and bbox prompts.
    """

    if not hasattr(seg_instance, "img"):
        raise AttributeError("seg_instance has no attribute 'img'.")

    if not hasattr(seg_instance, "gt"):
        raise AttributeError("seg_instance has no attribute 'gt'.")

    if not hasattr(seg_instance, "predicted_seg"):
        raise AttributeError(
            "seg_instance has no attribute 'predicted_seg'. "
            "Run seg_instance.run_segmentation() first."
        )

    img = seg_instance.img
    gt = seg_instance.gt.astype(bool)
    pred = seg_instance.predicted_seg.astype(bool)

    prompts_by_slice = getattr(seg_instance, "prompts_by_slice", {})

    if img.shape != gt.shape or img.shape != pred.shape:
        raise ValueError(
            f"Shape mismatch: img={img.shape}, gt={gt.shape}, pred={pred.shape}"
        )

    n_slices = img.shape[0]

    color_gt = "#57CC99"       # bright muted green
    color_dense = "#4A90E2"    # blue
    color_pred = "#F4A261"     # orange

    gt_cmap = ListedColormap([(0, 0, 0, 0), color_gt])
    dense_cmap = ListedColormap([(0, 0, 0, 0), color_dense])
    pred_cmap = ListedColormap([(0, 0, 0, 0), color_pred])

    def unpack_prompt(prompt):
        if isinstance(prompt, dict):
            return (
                prompt.get("points", None),
                prompt.get("point_labels", None),
                prompt.get("bbox", None),
                prompt.get("mask_input", None),
            )

        return prompt

    def shift_points(points, x0, y0):
        return np.column_stack([
            points[:, 0] - x0,
            points[:, 1] - y0,
        ])

    def _plot(
        slice_idx,
        zoom,
        show_ground_truth,
        show_dense_mask,
        show_updated_segmentation,
        show_point_prompts,
        show_bbox_prompts,
    ):
        z = slice_idx
        H, W = img[z].shape

        prompt_data = prompts_by_slice.get(z, None)

        if zoom:
            h_half = int(H * zoom_fraction / 2)
            w_half = int(W * zoom_fraction / 2)

            fg = gt[z] | pred[z]

            if prompt_data is not None:
                points, point_labels, bbox, mask_input = unpack_prompt(prompt_data)

                if mask_input is not None:
                    fg = fg | mask_input.astype(bool)

                if points is not None and len(points) > 0:
                    points_arr = np.asarray(points)
                    px = points_arr[:, 0]
                    py = points_arr[:, 1]

                    prompt_mask = np.zeros_like(fg, dtype=bool)
                    valid = (
                        (py >= 0) & (py < H) &
                        (px >= 0) & (px < W)
                    )

                    prompt_mask[
                        py[valid].astype(int),
                        px[valid].astype(int),
                    ] = True

                    fg = fg | prompt_mask

            if fg.any():
                ys, xs = np.where(fg)
                cy = int(np.round(np.mean(ys)))
                cx = int(np.round(np.mean(xs)))
            else:
                cy, cx = H // 2, W // 2

            y0 = max(0, cy - h_half)
            y1 = min(H, cy + h_half)
            x0 = max(0, cx - w_half)
            x1 = min(W, cx + w_half)
        else:
            y0, y1 = 0, H
            x0, x1 = 0, W

        img_crop = img[z][y0:y1, x0:x1]
        gt_crop = gt[z][y0:y1, x0:x1]
        pred_crop = pred[z][y0:y1, x0:x1]

        fig, ax = plt.subplots(figsize=figsize)

        ax.imshow(img_crop, cmap="gray")

        if show_ground_truth:
            gt_overlay = np.ma.masked_where(~gt_crop, gt_crop.astype(np.uint8))
            ax.imshow(gt_overlay, cmap=gt_cmap, alpha=alpha_gt, vmin=0, vmax=1)

        if show_dense_mask:
            if prompt_data is not None:
                _, _, _, mask_input = unpack_prompt(prompt_data)

                if mask_input is not None:
                    dense_crop = mask_input[y0:y1, x0:x1].astype(bool)
                    dense_overlay = np.ma.masked_where(
                        ~dense_crop,
                        dense_crop.astype(np.uint8),
                    )
                    ax.imshow(
                        dense_overlay,
                        cmap=dense_cmap,
                        alpha=alpha_dense_mask,
                        vmin=0,
                        vmax=1,
                    )

        if show_updated_segmentation:
            pred_overlay = np.ma.masked_where(~pred_crop, pred_crop.astype(np.uint8))
            ax.imshow(pred_overlay, cmap=pred_cmap, alpha=alpha_seg, vmin=0, vmax=1)

        if prompt_data is not None:
            points, point_labels, bbox, mask_input = unpack_prompt(prompt_data)

            if show_point_prompts and points is not None and point_labels is not None:
                points = np.asarray(points)
                point_labels = np.asarray(point_labels)

                if len(points) > 0:
                    pos = points[point_labels == 1]
                    neg = points[point_labels == 0]

                    if len(pos) > 0:
                        pos = shift_points(pos, x0, y0)
                        ax.scatter(
                            pos[:, 0],
                            pos[:, 1],
                            s=20,
                            c="lightgreen",
                            marker="+",
                            linewidths=1.5,
                            label="positive prompt",
                        )

                    if len(neg) > 0:
                        neg = shift_points(neg, x0, y0)
                        ax.scatter(
                            neg[:, 0],
                            neg[:, 1],
                            s=20,
                            c="lightcoral",
                            marker="x",
                            linewidths=1.5,
                            label="negative prompt",
                        )

            if show_bbox_prompts and bbox is not None:
                boxes = np.asarray(bbox)

                if boxes.ndim == 1:
                    boxes = boxes[None, :]

                for i, (bx0, by0, bx1, by1) in enumerate(boxes):
                    rect = Rectangle(
                        (bx0 - x0, by0 - y0),
                        bx1 - bx0,
                        by1 - by0,
                        fill=False,
                        edgecolor="cyan",
                        linewidth=1.2,
                        label="bbox prompt" if i == 0 else None,
                    )
                    ax.add_patch(rect)

        title = f"Slice {z}"

        if show_ground_truth:
            title += f" | GT: {int(gt_crop.sum())}"

        if show_dense_mask:
            title += " | Dense mask"

        if show_updated_segmentation:
            title += f" | Seg: {int(pred_crop.sum())}"

        ax.set_title(title)
        ax.set_axis_off()

        handles = []

        if show_ground_truth:
            handles.append(mpatches.Patch(color=color_gt, label="Ground truth"))

        if show_dense_mask:
            handles.append(mpatches.Patch(color=color_dense, label="Dense mask prompt"))

        if show_updated_segmentation:
            handles.append(mpatches.Patch(color=color_pred, label="Updated segmentation"))

        existing_handles, existing_labels = ax.get_legend_handles_labels()
        handles.extend(existing_handles)

        if len(handles) > 0:
            labels = [h.get_label() for h in handles]
            unique = dict(zip(labels, handles))
            ax.legend(unique.values(), unique.keys(), loc="upper right", framealpha=0.85)

        plt.tight_layout()
        plt.show()

    interact(
        _plot,
        slice_idx=IntSlider(
            min=0,
            max=n_slices - 1,
            step=1,
            value=0,
            description="slice",
        ),
        zoom=Checkbox(value=False, description="zoom"),
        show_ground_truth=Checkbox(value=True, description="ground truth"),
        show_dense_mask=Checkbox(value=True, description="dense mask"),
        show_updated_segmentation=Checkbox(value=True, description="segmentation"),
        show_point_prompts=Checkbox(value=True, description="points"),
        show_bbox_prompts=Checkbox(value=True, description="bbox"),
    )

In [ ]:
plot_segmentation_overlays_interactive(seg_handler)

interactive(children=(IntSlider(value=0, description='slice', max=87), Checkbox(value=False, description='zoom…

In [10]:
eval_handler = Evaluator(segmentation=seg_handler)
metrics = eval_handler.compute_all(surface_dice_tol=1.0)

nn_unet_eval = Evaluator(
    pred=seg_handler.mask,
    gt=seg_handler.gt,
    spacing=seg_handler.img_spacing,
)
nn_unet_metrics = nn_unet_eval.compute_all(surface_dice_tol=1.0)


print("\n" + "=" * 95)
print(f"{'Metric':<40} {'nnUNet Dense Mask':>22} {'Updated Segmentation':>28}")
print("=" * 95)

for key in nn_unet_metrics.keys():

    nn_val = nn_unet_metrics[key]
    new_val = metrics[key]

    print(
        f"{key:<40} "
        f"{nn_val:>22.4f} "
        f"{new_val:>28.4f}"
    )

print("=" * 95)


Metric                                        nnUNet Dense Mask         Updated Segmentation
HD_mm                                                    7.1981                      89.3406
HD95_mm                                                  4.5316                      87.5163
MSD_mm                                                   1.2760                      32.7504
ASSD_mm                                                  1.2589                      16.9602
SurfaceDice@1.0mm                                        0.5724                       0.3892
CentroidDistance_mm                                      2.2639                      23.7409
PredictionVolume_mm3                                 73044.4525                   82050.2190
GroundTruthVolume_mm3                                66441.9087                   66441.9087
AbsVolumeDifference_mm3                               6602.5438                   15608.3103
RelativeVolumeDifference_percent                         9.9373      